In [1]:
# ============================================================
# 02_kpi_tables_export.ipynb
# KPI Exports (Reads enriched file only)
# ============================================================

import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

ANALYSIS_DATE = pd.Timestamp("2026-06-03")

ENRICHED_PATH = "D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_purchase_data_enriched.csv"

os.makedirs("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data", exist_ok=True)

# -----------------------------
# Load enriched data
# -----------------------------
df = pd.read_csv(ENRICHED_PATH)
print("Enriched shape:", df.shape)
display(df.head())

# Parse dates
df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")
df["Last Purchase Date"] = pd.to_datetime(df["Last Purchase Date"], errors="coerce")

# Ensure needed columns exist
if "Discount Flag" not in df.columns:
    df["Discount Flag"] = np.where(df["Discount Used"].astype(str).str.upper() == "YES", 1, 0)

if "Order Month" not in df.columns:
    df["Order Month"] = df["Purchase Date"].dt.to_period("M").astype(str)

# -----------------------------
# KPI Summary (prints only)
# -----------------------------
total_revenue = df["Total Purchase Value"].sum()
total_orders = df.shape[0]
total_customers = df["Customer ID"].nunique()
aov = total_revenue / total_orders if total_orders else 0

orders_per_customer = df.groupby("Customer ID")["Purchase Date"].count()
repeat_customers = int((orders_per_customer >= 2).sum())
repeat_rate = repeat_customers / total_customers if total_customers else 0

avg_orders_per_customer = total_orders / total_customers if total_customers else 0
basic_clv_12 = aov * avg_orders_per_customer * 12

print("\n--- KPI SUMMARY ---")
print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"AOV: {aov:,.2f}")
print(f"Repeat Purchase Rate: {repeat_rate:.2%}")
print(f"Basic CLV (12 mo): {basic_clv_12:,.2f}")

# -----------------------------
# Aggregated KPI Tables
# -----------------------------

# Customer KPIs
customer_kpis = (df
    .groupby(["Customer ID","Customer Name","Gender","Age Group","City","Region","Occupation","Customer Segment","Loyalty Status"], as_index=False)
    .agg(
        Total_Revenue=("Total Purchase Value","sum"),
        Orders=("Purchase Date","count"),
        Total_Quantity=("Quantity Purchased","sum"),
        Avg_Unit_Price=("Unit Price","mean"),
        Last_Purchase=("Last Purchase Date","max"),
        Discount_Usage_Rate=("Discount Flag","mean")
    )
)
customer_kpis["AOV"] = customer_kpis["Total_Revenue"] / customer_kpis["Orders"]
customer_kpis["Days Since Last Purchase"] = (ANALYSIS_DATE - customer_kpis["Last_Purchase"]).dt.days
customer_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/customer_kpis.csv", index=False)

# Monthly KPIs
monthly_kpis = (df.groupby("Order Month", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
monthly_kpis["AOV"] = monthly_kpis["Revenue"] / monthly_kpis["Orders"]
monthly_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/monthly_kpis.csv", index=False)

# Category KPIs
category_kpis = (df.groupby("Product Category", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Quantity=("Quantity Purchased","sum"), Customers=("Customer ID","nunique"))
)
category_kpis["AOV"] = category_kpis["Revenue"] / category_kpis["Orders"]
category_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/category_kpis.csv", index=False)

# Segment KPIs
segment_kpis = (df.groupby("Customer Segment", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
segment_kpis["AOV"] = segment_kpis["Revenue"] / segment_kpis["Orders"]
segment_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/segment_kpis.csv", index=False)

# Region KPIs
region_kpis = (df.groupby("Region", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
region_kpis["AOV"] = region_kpis["Revenue"] / region_kpis["Orders"]
region_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/region_kpis.csv", index=False)

# Payment KPIs
payment_kpis = (df.groupby("Payment Method", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
payment_kpis["AOV"] = payment_kpis["Revenue"] / payment_kpis["Orders"]
payment_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/payment_kpis.csv", index=False)

# Channel KPIs
channel_kpis = (df.groupby("Purchase Channel", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
channel_kpis["AOV"] = channel_kpis["Revenue"] / channel_kpis["Orders"]
channel_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/channel_kpis.csv", index=False)

# Loyalty KPIs
loyalty_kpis = (df.groupby("Loyalty Status", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
loyalty_kpis["AOV"] = loyalty_kpis["Revenue"] / loyalty_kpis["Orders"]
loyalty_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/loyalty_kpis.csv", index=False)

# Discount KPIs
discount_kpis = (df.groupby("Discount Used", as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Customers=("Customer ID","nunique"))
)
discount_kpis["AOV"] = discount_kpis["Revenue"] / discount_kpis["Orders"]
discount_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/discount_kpis.csv", index=False)

# Recency KPIs
df["Days Since Last Purchase"] = (ANALYSIS_DATE - df["Last Purchase Date"]).dt.days
def recency_bucket(x):
    if pd.isna(x): return "Unknown"
    if x <= 30: return "0-30 days"
    if x <= 60: return "31-60 days"
    if x <= 90: return "61-90 days"
    return "90+ days"

df["Recency Bucket"] = df["Days Since Last Purchase"].apply(recency_bucket)

recency_kpis = (df.groupby("Recency Bucket", as_index=False)
    .agg(Customers=("Customer ID","nunique"), Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"))
)
recency_kpis["AOV"] = recency_kpis["Revenue"] / recency_kpis["Orders"]
recency_kpis.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/recency_kpis.csv", index=False)

# Top 10 Customers
top_10_customers = (df.groupby(["Customer ID","Customer Name"], as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Quantity=("Quantity Purchased","sum"))
    .sort_values("Revenue", ascending=False)
    .head(10)
)
top_10_customers.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/top_10_customers.csv", index=False)

# Top 10 Products
top_10_products = (df.groupby(["Product Category","Product Name"], as_index=False)
    .agg(Revenue=("Total Purchase Value","sum"), Orders=("Purchase Date","count"), Quantity=("Quantity Purchased","sum"))
    .sort_values("Revenue", ascending=False)
    .head(10)
)
top_10_products.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/top_10_products.csv", index=False)

# At-risk customers list (simple)
rev_threshold = customer_kpis["Total_Revenue"].quantile(0.70)
at_risk = customer_kpis[(customer_kpis["Total_Revenue"] >= rev_threshold) & (customer_kpis["Days Since Last Purchase"] > 30)] \
            .sort_values(["Total_Revenue","Days Since Last Purchase"], ascending=[False, False])
at_risk.to_csv("D:/data/Documents/MBA/IIP Skills upgrade courses/Projects/Data Analyst/customer-purchase-pattern-analyzer/data/at_risk_customers.csv", index=False)

print("\nSaved all KPI tables into ../data/")
print("Files created:")
print("- customer_kpis.csv, monthly_kpis.csv, category_kpis.csv, segment_kpis.csv, region_kpis.csv")
print("- payment_kpis.csv, channel_kpis.csv, loyalty_kpis.csv, discount_kpis.csv, recency_kpis.csv")
print("- top_10_customers.csv, top_10_products.csv, at_risk_customers.csv")

display(customer_kpis.sort_values("Total_Revenue", ascending=False).head(10))
display(recency_kpis)

Enriched shape: (500, 28)


,Customer ID,Customer Name,Age,Gender,City,Region,Occupation,Product Category,Product Name,Purchase Date,Quantity Purchased,Unit Price,Total Purchase Value,Payment Method,Purchase Channel,Customer Segment,Loyalty Status,Discount Used,Purchase Frequency,Last Purchase Date,Total Mismatch Flag,Outlier Flag,Order Month,Order Quarter,Order Weekday,Age Group,Discount Flag,Days Since Last Purchase
0,CUST0130,Leah Sharma,19,Female,Detroit,Midwest,Director,Groceries,Basmati Rice 5lb,2025-12-29,3,18.09,54.27,Credit Card,Website,Budget Buyers,Silver,No,Medium,2026-01-02,0,0,2025-12,2025Q4,Monday,18-24,0,152
1,CUST0175,Vivaan Moore,52,Female,San Diego,West,Data Analyst,Beverages,Premium Coffee Pods,2026-05-21,1,14.30,14.30,Cash,Store,High Value,Gold,No,High,2026-06-03,0,0,2026-05,2026Q2,Thursday,45-54,0,0
2,CUST0084,Muhammad Davis,49,Female,San Diego,West,Teacher,Snacks,Trail Mix,2026-03-02,3,5.39,16.17,UPI/Wallet,Store,High Value,Gold,No,High,2026-03-29,0,0,2026-03,2026Q1,Monday,45-54,0,66
3,CUST0022,Noah Taylor,34,Male,Orlando,South,Data Scientist,Personal Care,Shampoo,2025-12-18,2,6.49,12.98,Credit Card,Mobile App,Young Professionals,Silver,Yes,Medium,2026-02-08,0,0,2025-12,2025Q4,Thursday,25-34,1,115
4,CUST0027,Benjamin Das,46,Female,Newark,Northeast,Nurse,Home Care,Paper Towels,2026-05-07,1,3.37,3.37,Net Banking,Mobile App,Family Shoppers,NaN,No,Medium,2026-06-03,0,0,2026-05,2026Q2,Thursday,45-54,0,0



--- KPI SUMMARY ---
Total Revenue: 12,851.96
Total Orders: 500
Total Customers: 170
AOV: 25.70
Repeat Purchase Rate: 84.71%
Basic CLV (12 mo): 907.20

Saved all KPI tables into ../data/
Files created:
- customer_kpis.csv, monthly_kpis.csv, category_kpis.csv, segment_kpis.csv, region_kpis.csv
- payment_kpis.csv, channel_kpis.csv, loyalty_kpis.csv, discount_kpis.csv, recency_kpis.csv
- top_10_customers.csv, top_10_products.csv, at_risk_customers.csv


,Customer ID,Customer Name,Gender,Age Group,City,Region,Occupation,Customer Segment,Loyalty Status,Total_Revenue,Orders,Total_Quantity,Avg_Unit_Price,Last_Purchase,Discount_Usage_Rate,AOV,Days Since Last Purchase
43,CUST0072,Harper Davis,Male,25-34,Nashville,South,Data Scientist,High Value,Silver,272.81,6,16,16.655000,2026-06-03,0.333333,45.468333,0
53,CUST0084,Muhammad Davis,Female,45-54,San Diego,West,Teacher,High Value,Gold,265.48,8,22,11.918750,2026-06-03,0.625000,33.185000,0
64,CUST0106,Liam Jackson,Female,25-34,Detroit,Midwest,Sales Executive,Young Professionals,Silver,248.11,4,17,13.275000,2026-05-05,0.250000,62.027500,29
105,CUST0172,Leah Perez,Female,45-54,New York,Northeast,HR Manager,Family Shoppers,Silver,199.57,6,12,17.775000,2026-05-12,0.333333,33.261667,22
100,CUST0166,Ella White,Female,45-54,Atlanta,South,Software Engineer,High Value,Gold,194.16,5,15,10.910000,2026-06-03,0.400000,38.832000,0
56,CUST0089,Lily Moore,Male,55+,Philadelphia,Northeast,Operations Manager,High Value,Gold,183.05,5,11,15.978000,2026-05-28,0.400000,36.610000,6
97,CUST0158,Muhammad Davis,Female,35-44,Orlando,South,Data Scientist,Young Professionals,Gold,159.63,3,10,18.083333,2026-06-03,0.000000,53.210000,0
85,CUST0136,Ava Thomas,Female,25-34,Orlando,South,Director,High Value,Silver,157.85,4,8,19.182500,2026-06-03,0.000000,39.462500,0
66,CUST0108,Lily Rodriguez,Female,35-44,Boston,Northeast,HR Manager,Young Professionals,Gold,157.37,4,13,11.815000,2026-06-03,0.250000,39.342500,0
23,CUST0041,Krishna Wilson,Male,25-34,Los Angeles,West,Nurse,High Value,Gold,147.17,4,6,24.657500,2026-06-03,0.000000,36.792500,0


,Recency Bucket,Customers,Revenue,Orders,AOV
0,0-30 days,123,4690.22,191,24.556126
1,31-60 days,56,1919.27,67,28.645821
2,61-90 days,68,2242.53,81,27.685556
3,90+ days,103,3999.94,161,24.844348
